<a href="https://colab.research.google.com/github/Serajummunira/ip-intelligence-enrichment/blob/main/ip_enrichment_pipeline_rdap_geoip.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pandas requests

In [ ]:
import pandas as pd
import requests
import socket
import os
import time

In [ ]:
# Optional (recommended) Set api kEY
from getpass import getpass
import os

# Enter API keys securely (not saved in notebook)
os.environ["IPINFO_TOKEN"] = getpass("Enter IPinfo API key: ")
os.environ["ABUSEIPDB_API_KEY"] = getpass("Enter AbuseIPDB API key: ")

In [ ]:
#Upload CSV file
from google.colab import files
uploaded = files.upload()

ARIN RDAP (ownership data)

In [ ]:
def get_arin_data(ip):
    try:
        url = f"https://rdap.arin.net/registry/ip/{ip}"
        res = requests.get(url, timeout=10)
        if res.status_code != 200:
            return {}

        data = res.json()

        result = {
            "arin_handle": data.get("handle"),
            "arin_name": data.get("name"),
            "arin_type": data.get("type"),
            "arin_start_ip": data.get("startAddress"),
            "arin_end_ip": data.get("endAddress"),
            "arin_country": data.get("country"),
            "arin_parent": data.get("parentHandle")
        }

        # Extract abuse email if available
        abuse_email = None
        for ent in data.get("entities", []):
            roles = ent.get("roles", [])
            if "abuse" in roles:
                vcard = ent.get("vcardArray", [])
                if len(vcard) > 1:
                    for item in vcard[1]:
                        if item[0] == "email":
                            abuse_email = item[3]

        result["arin_abuse_email"] = abuse_email
        return result

    except:
        return {}

IPinfo (Geo + ASN + ISP)

In [ ]:
def get_ipinfo_data(ip):
    try:
        token = os.environ["IPINFO_TOKEN"] = getpass("Enter IPinfo API key: ")
        url = f"https://ipinfo.io/{ip}/json"
        if token:
            url += f"?token={token}"

        res = requests.get(url, timeout=10)
        if res.status_code != 200:
            return {}

        data = res.json()

        return {
            "geo_country": data.get("country"),
            "geo_region": data.get("region"),
            "geo_city": data.get("city"),
            "geo_location": data.get("loc"),
            "org": data.get("org"),
            "hostname": data.get("hostname")
        }

    except:
        return {}

AbuseIPDB (threat intelligence)

In [ ]:
def get_abuseipdb_data(ip):
    try:
        api_key = os.environ["IPINFO_TOKEN"] = getpass("Enter IPinfo API key: ")
        if not api_key:
            return {}

        url = "https://api.abuseipdb.com/api/v2/check"
        headers = {
            "Key": api_key,
            "Accept": "application/json"
        }
        params = {
            "ipAddress": ip,
            "maxAgeInDays": 90
        }

        res = requests.get(url, headers=headers, params=params, timeout=10)
        if res.status_code != 200:
            return {}

        data = res.json().get("data", {})

        return {
            "abuse_score": data.get("abuseConfidenceScore"),
            "total_reports": data.get("totalReports"),
            "isp": data.get("isp"),
            "usage_type": data.get("usageType")
        }

    except:
        return {}

Reverse DNS

In [ ]:
def get_reverse_dns(ip):
    try:
        return socket.gethostbyaddr(ip)[0]
    except:
        return None

Run enrichment

In [ ]:
df = pd.read_csv("/content/10000_ip_addresses.csv")

results = []

for ip in df["ip"]:
    print(f"Processing {ip}...")

    record = {"ip": ip}

    # Merge all data sources
    record.update(get_arin_data(ip))
    record.update(get_ipinfo_data(ip))
    record.update(get_abuseipdb_data(ip))

    record["reverse_dns"] = get_reverse_dns(ip)

    results.append(record)

    time.sleep(1)  # avoid rate limits

enriched_df = pd.DataFrame(results)
enriched_df.head()

Save output

In [ ]:
enriched_df.to_csv("geoip_10000_output.csv", index=False)

from google.colab import files
files.download("geoip_10000_output.csv")